In [ ]:
# after extract all the surface patches from step1
# In step 2, we perform all the prediction
# condition: 1 use all the positive patches 2 negative patch residue1 SasaRate > 0.2 and residue 2 SasaRate > 0.2

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install biopython --target=$nb_path

In [ ]:
%pip install ipywidgets

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import random

import itertools
import math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
#@title load cnn model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(    # input shape (1,15,20)
            nn.Conv2d(
                in_channels=1,         # input height: layers
                out_channels=16,       # n fileters
                kernel_size=(3,3),         # filter size
                stride=1,              # filter movement/step
                padding=(1,1),             # want same width and length of this image after Conv2d
            ),                         # ==> (16, 15, 20)
            nn.BatchNorm2d(16),        #applying batch norm
            nn.ReLU(),                 # activation
            nn.MaxPool2d(kernel_size=(3,2)), # choose max value in 2x2 area, output shape
        )                              # output shape (16,5,10)
        self.conv2 = nn.Sequential(    # input shape (16,5,10)
            nn.Conv2d(16,32,(2,2),1,(1,1)),    # output shape(32,5,10)
            nn.BatchNorm2d(32),        #applying batch norm
            nn.ReLU(),                 # activation
            nn.MaxPool2d(2),           # output shape (32,3,5)
        )
        self.fc1 = nn.Linear(32 * 3 * 5, 128)
        self.fc2 = nn.Linear(128, 32)
        self.out = nn.Linear(32, 1) # fully connected layer, output 10 classes

    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)              # (batch, 32, 7, 7)
        x = x.view(x.size(0), -1)      # flatten the output of conv2 to (batch_size, 32*7*7)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        output = torch.sigmoid(self.out(x))
        return output

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cnn = CNN().to(device)

In [ ]:
"""
from Bio import PDB

import numpy as np
import matplotlib.pyplot as plt

def find_antigen_chains(pdb_file, antibody_heavy_chain, antibody_light_chain):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("structure", pdb_file)

    antigen_chains = []
    for model in structure:
        for chain in model:
            if chain.id not in [antibody_heavy_chain, antibody_light_chain]:
                antigen_chains.append(chain.id)
    return antigen_chains

def find_interactions(pdb_file, antibody_chains, antigen_chains, distance_cutoff=4.0):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)

    antibody_atoms = []
    for chain in antibody_chains:
        antibody_atoms.extend([atom for atom in structure[0][chain].get_atoms()])

    interacting_antigen_atoms = set()

    for antigen_chain in antigen_chains:
        antigen_atoms = [atom for atom in structure[0][antigen_chain].get_atoms()]
        for antibody_atom in antibody_atoms:
            for antigen_atom in antigen_atoms:
                distance = antibody_atom - antigen_atom
                if distance <= distance_cutoff:
                    interacting_antigen_atoms.add(antigen_atom)

    return interacting_antigen_atoms

def calculate_interaction_radius(interacting_atoms):
    if not interacting_atoms:
        return None

    coords = [atom.get_coord() for atom in interacting_atoms]
    coords_array = np.array(coords)

    # Calculate centroid
    centroid = np.mean(coords_array, axis=0)

    # Calculate distances from each interacting atom to the centroid
    distances = np.linalg.norm(coords_array - centroid, axis=1)

    # Calculate average distance as radius estimate
    avg_radius = np.mean(distances)

    return avg_radius

def main(folder_path, antibody_heavy_chain, antibody_light_chain):
    pdb_files = [f for f in os.listdir(folder_path) if f.endswith('.pdb')]
    interaction_counts = []
    radii = []

    for pdb_file in pdb_files:

            pdb_path = os.path.join(folder_path, pdb_file)
            antigen_chains = find_antigen_chains(pdb_path, antibody_heavy_chain, antibody_light_chain)
            interacting_atoms = find_interactions(pdb_path, [antibody_heavy_chain, antibody_light_chain], antigen_chains)
            interaction_counts.append(len(interacting_atoms))

            radius = calculate_interaction_radius(interacting_atoms)
            if radius:
                radii.append(radius)


    return interaction_counts, radii

# Process each fold's training set
for fold in range(1, 6):

    if __name__ == "__main__":

        folder_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v4/" + f'fold_{fold}/data/abdb_data'
        antibody_heavy_chain = "H"
        antibody_light_chain = "L"

        interaction_counts, radii = main(folder_path, antibody_heavy_chain, antibody_light_chain)

        if radii:
            avg_radius = np.mean(radii)
            print(f"\nAverage radius across all proteins: {avg_radius:.2f} Angstroms")
        else:
            print("No valid radii calculated.")
"""

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.cluster import DBSCAN

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Create a list to store DataFrames from each fold
all_fold_dfs = []

# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df['prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)


    merged_df['prediction'] = pd.to_numeric(merged_df['prediction'], errors='coerce')
    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['prediction'].max()
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(normalize_group)
    # merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.nan)
    merged_df.drop('prediction', axis=1, inplace=True)



    '''
    def top_clusters_with_highest_pred(group,distance_threshold,num_clusters):
        # Extract coordinates
        coords = group[['Cax', 'Cay', 'Caz']].values

        # Use DBSCAN to cluster residues based on the distance threshold
        clustering = DBSCAN(eps=distance_threshold, min_samples=1).fit(coords)
        group['cluster'] = clustering.labels_

        # Calculate the prediction sum for each cluster
        cluster_pred_sums = group.groupby('cluster')['pred'].max().sort_values(ascending=False)

        # Select the top clusters based on the sum of predictions
        top_clusters = cluster_pred_sums.head(num_clusters).index

        # Set 'pred' to 1 for residues in the top clusters, 0 otherwise
        group['pred'] = group['cluster'].apply(lambda x: 1 if x in top_clusters else 0)

        # Drop the 'cluster' column
        group = group.drop(columns=['cluster'])

        return group

    distance_threshold = 4.5
    num_clusters = 3

    # Apply the function to each group in the DataFrame
    merged_df = merged_df.groupby('name').apply(lambda x: top_clusters_with_highest_pred(x,distance_threshold,num_clusters)).reset_index(drop=True)
    '''



    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Collect the DataFrame
    all_fold_dfs.append(merged_df)

# Combine all DataFrames
combined_df = pd.concat(all_fold_dfs, ignore_index=True)

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.cluster import DBSCAN

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Create a list to store DataFrames from each fold
all_fold_dfs = []

# Process each fold's training set
for fold in range(1, 6):
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Load testing data and metadata
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')

    # Load patches and convert to tensor
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)

    # Load patch metadata (protein names and residue indices)
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Parse or load existing metadata
    if not os.path.exists(output_file_path):
        # Parsing code from original implementation
        protein_names = []
        patch_indices = []
        with open(input_file_path, 'r') as file:
            lines = file.readlines()
            current_protein_name = None
            for line in lines:
                line = line.strip()
                if line.startswith('Patches for protein'):
                    current_protein_name = line.split(' ')[3][:-1]
                elif line.startswith('indices:'):
                    indices = line.split('[')[1].split(']')[0]
                    protein_names.append(current_protein_name)
                    patch_indices.append(indices)
        df_patch = pd.DataFrame({'Protein_Name': protein_names, 'Patch_Indices': patch_indices})
        df_patch.to_csv(output_file_path, index=False)
    else:
        df_patch = pd.read_csv(output_file_path)

    # 3. Group patches by protein
    grouped = df_patch.groupby('Protein_Name')

    # 4. Process each protein individually
    all_predictions = []
    for protein_name, group in grouped:
        # Get indices for this protein's patches
        indices = group.index.tolist()
        protein_patches = testing_patches_tensor[indices]

        # Create DataLoader for this protein
        protein_loader = DataLoader(protein_patches, batch_size=BATCH_SIZE, shuffle=False)

        # Make predictions
        protein_predictions = []
        with torch.no_grad():
            for batch in protein_loader:
                outputs = model_cnn(batch)
                outputs = outputs.squeeze().cpu().numpy()
                protein_predictions.extend(outputs)

        # Store predictions with metadata
        protein_df = group.copy()
        protein_df['Predictions'] = protein_predictions
        all_predictions.append(protein_df)

    # Combine all protein predictions
    df_patch = pd.concat(all_predictions)

    # 5. Merge predictions with original data
    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')
    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))
    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Convert patch indices to integer list
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Merge predictions
    merged_df = pd.merge(filtered_combined_df,
                        df_patch_long,
                        left_on=['name', 'ResID'],
                        right_on=['Protein_Name', 'Patch_Indices'],
                        how='left')

    # Post-processing
    merged_df['prediction'] = merged_df['Predictions']
    merged_df = merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'])

    # Normalize predictions per protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(
        lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)

    # Aggregate results
    df_groupby_cols = merged_df.columns.tolist()[:-2]  # Exclude prediction and pred columns
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()


    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Collect the DataFrame
    all_fold_dfs.append(merged_df)

# Combine all DataFrames
combined_df = pd.concat(all_fold_dfs, ignore_index=True)

In [ ]:
# Save the final combined DataFrame to a CSV file
combined_df.to_csv(os.path.join(save_dir, 'all_data_combined_pred.csv'), index=False)

# Display the first few rows of the combined DataFrame
print(combined_df.head())

In [ ]:
print(combined_df[combined_df['SaSaRate'] < 0.1])

In [ ]:
merged_df_filtered = combined_df.dropna(subset=['pred'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set up the matplotlib figure
plt.figure(figsize=(10, 6))

# Plot the distribution of pred values for each Status
sns.histplot(data=merged_df_filtered, x='pred', hue='Status', element='step', stat="density", common_norm=False)

# Provide labels and title
plt.xlabel('Prediction Value')
plt.ylabel('Density')
plt.title('Distribution of Prediction Values by Status')
plt.legend(title='Status')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get true binary labels
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Get prediction scores
pred_scores = merged_df_filtered[merged_df_filtered['SaSaRate'] >= 0.1]['pred'].values

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(true_labels, pred_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) for CNN')
plt.legend(loc="lower right")
plt.show()


In [ ]:
#@title find the best cut-off
from sklearn.metrics import roc_curve

# Convert 'Status' column to binary values
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Get false positive rate, true positive rate and thresholds using roc_curve
fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

# Calculate Youden's J statistic for each threshold
J = tpr - fpr

# Get the index of the maximum J value
idx = np.argmax(J)

# Get the best threshold
best_threshold = thresholds[idx]

print(f"Best cut-off (threshold): {best_threshold:.3f}")


In [ ]:
# Convert 'Status' column to binary values
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Threshold 'pred' values at 0.5 to get predicted classes
predicted_labels = np.where(merged_df_filtered['pred'] >= 0.5, 1, 0)

# Calculate accuracy
accuracy = np.mean(true_labels == predicted_labels)

print(f"Accuracy: {accuracy*100:.2f}%")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Convert 'Status' column to binary values
true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

# Calculate metrics
accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
precision = precision_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
f1 = f1_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
auc = roc_auc_score(true_labels, merged_df_filtered['pred'])

# Display metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Best Cutoff (Threshold): {best_threshold:.3f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(true_labels, pred_scores)
roc_auc = auc(fpr, tpr)
print(f'ROC-AUC: {roc_auc:.2f}')

# Compute Balanced Accuracy (BAC)
bac = balanced_accuracy_score(true_labels, pred_scores)
print(f'BAC: {bac:.2f}')

# Compute Matthews Correlation Coefficient (MCC)
mcc = matthews_corrcoef(true_labels, pred_scores)
print(f'MCC: {mcc:.2f}')

# Compute Precision-Recall curve and AUC
precision, recall, _ = precision_recall_curve(true_labels, pred_scores)
pr_auc = auc(recall, precision)
print(f'PR-AUC: {pr_auc:.2f}')

# Compute Nantigens (number of unique antigens)
nantigens = merged_df_filtered['name'].nunique()
print(f'Nantigens: {nantigens}')

# Compute Fpredicted (mean fraction of surface residues predicted as epitopes)
f_predicted = np.mean(pred_scores) * 100
print(f'Fpredicted: {f_predicted:.2f}%')

In [ ]:
import numpy as np

# Calculate AUC for each protein
protein_names = merged_df_filtered['name'].unique()
auc_list = []

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        auc = roc_auc_score(true_labels, protein_df['pred'])
        auc_list.append(auc)

# Plot AUC distribution
plt.hist(auc_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('AUC')
plt.ylabel('Frequency')
plt.title('Distribution of AUC for each protein without large patch prediction')
plt.grid(True)
plt.show()

print('The mean AUC of protein is', np.mean(auc_list))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, recall_score

# Calculate recall for each protein
protein_names = merged_df_filtered['name'].unique()
recall_list = []

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['pred'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        recall = recall_score(true_labels, (protein_df['pred'] >= best_threshold).astype(int))
        recall_list.append(recall)

# Plot recall distribution
plt.hist(recall_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Recall')
plt.ylabel('Frequency')
plt.title('Distribution of recall for each protein without large patch prediction')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, accuracy_score

# Calculate accuracy for each protein
protein_names = merged_df_filtered['name'].unique()
accuracy_list = []

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['pred'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        accuracy = accuracy_score(true_labels, (protein_df['pred'] >= best_threshold).astype(int))
        accuracy_list.append(accuracy)

# Plot accuracy distribution
plt.hist(accuracy_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.title('Distribution of accuracy for each protein without large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
# Calculate AUC for each protein
protein_names = merged_df_filtered['name'].unique()
auc_list = []

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        auc = roc_auc_score(true_labels, protein_df['pred'])
        auc_list.append(auc)

# Plot AUC distribution
plt.hist(auc_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('AUC')
plt.ylabel('Frequency')
plt.title('Distribution of AUC for each protein using large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, recall_score

# Calculate recall for each protein
protein_names = merged_df_filtered['name'].unique()
recall_list = []

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['pred'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        recall = recall_score(true_labels, (protein_df['pred'] >= best_threshold).astype(int))
        recall_list.append(recall)

# Plot recall distribution
plt.hist(recall_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Recall')
plt.ylabel('Frequency')
plt.title('Distribution of recall for each protein using large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, accuracy_score

# Calculate accuracy for each protein
protein_names = merged_df_filtered['name'].unique()
accuracy_list = []

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['pred'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        accuracy = accuracy_score(true_labels, (protein_df['pred'] >= best_threshold).astype(int))
        accuracy_list.append(accuracy)

# Plot accuracy distribution
plt.hist(accuracy_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.title('Distribution of accuracy for each protein using large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df.loc[:, 'prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)

    # Normalize the prediction values for each protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    merged_df.drop('prediction', axis=1, inplace=True)

    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()

    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Filter out rows where 'pred' is NaN
    merged_df_filtered = merged_df.dropna(subset=['pred'])

    merged_df_filtered.to_csv(os.path.join(save_dir, f'data_{fold}_combined_pred.csv'), index=False)

    # Get true binary labels
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get prediction scores
    pred_scores = merged_df_filtered['pred'].values

    # Compute ROC curve and AUC
    fpr, tpr, thresholds = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()


    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get false positive rate, true positive rate and thresholds using roc_curve
    fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

    # Calculate Youden's J statistic for each threshold
    J = tpr - fpr

    # Get the index of the maximum J value
    idx = np.argmax(J)

    # Get the best threshold
    best_threshold = thresholds[idx]

    # Calculate metrics
    accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    precision = precision_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    f1 = f1_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    roc_auc_value = roc_auc_score(true_labels, merged_df_filtered['pred'])

    # Display metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC: {roc_auc_value:.4f}")
    print(f"Best Cutoff (Threshold): {best_threshold:.3f}")

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score, roc_auc_score, accuracy_score

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df['prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)

    # Normalize the prediction values for each protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    merged_df.drop('prediction', axis=1, inplace=True)

    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()




    def top_residues_within_radius(group, avg_radius):
        group = group.sort_values('pred', ascending=False)
        top_10 = group.iloc[:10]
        top_10_coords = top_10[['Cax', 'Cay', 'Caz']].values
        all_coords = group[['Cax', 'Cay', 'Caz']].values

        max_pred_sum = 0
        best_coords = None

        for coord in top_10_coords:
            distances = np.sqrt(np.sum((all_coords - coord)**2, axis=1))
            current_group = group[distances <= avg_radius]
            current_pred_sum = current_group['pred'].sum()

            if current_pred_sum > max_pred_sum:
                max_pred_sum = current_pred_sum
                best_coords = current_group.index

        group['pred'] = 0

        if best_coords is not None:
            group.loc[best_coords, 'pred'] = 1

        return group

    avg_radius = 8.7

    merged_df = merged_df.groupby('name').apply(lambda x: top_residues_within_radius(x, avg_radius)).reset_index(drop=True)





    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Filter out rows where 'pred' is NaN
    merged_df_filtered = merged_df.dropna(subset=['pred'])

    # Get true binary labels
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get prediction scores
    pred_scores = merged_df_filtered['pred'].values

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()


    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get false positive rate, true positive rate and thresholds using roc_curve
    fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

    # Calculate Youden's J statistic for each threshold
    J = tpr - fpr

    # Get the index of the maximum J value
    idx = np.argmax(J)

    # Get the best threshold
    best_threshold = thresholds[idx]


    recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    print(f"Recall: {recall:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score, roc_auc_score, accuracy_score

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df['prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)

    # Normalize the prediction values for each protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    merged_df.drop('prediction', axis=1, inplace=True)

    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()




    def top_residues_within_radius(group, avg_radius):
        group = group.sort_values('pred', ascending=False)
        top_10 = group.iloc[:10]
        top_10_coords = top_10[['Cax', 'Cay', 'Caz']].values
        all_coords = group[['Cax', 'Cay', 'Caz']].values

        max_pred_sum = 0
        best_coords = None

        for coord in top_10_coords:
            distances = np.sqrt(np.sum((all_coords - coord)**2, axis=1))
            current_group = group[distances <= avg_radius]
            current_pred_sum = current_group['pred'].sum()

            if current_pred_sum > max_pred_sum:
                max_pred_sum = current_pred_sum
                best_coords = current_group.index

        group['pred'] = 0

        if best_coords is not None:
            group.loc[best_coords, 'pred'] = 1

        return group

    avg_radius = 10

    merged_df = merged_df.groupby('name').apply(lambda x: top_residues_within_radius(x, avg_radius)).reset_index(drop=True)





    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Filter out rows where 'pred' is NaN
    merged_df_filtered = merged_df.dropna(subset=['pred'])

    # Get true binary labels
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get prediction scores
    pred_scores = merged_df_filtered['pred'].values

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()


    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get false positive rate, true positive rate and thresholds using roc_curve
    fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

    # Calculate Youden's J statistic for each threshold
    J = tpr - fpr

    # Get the index of the maximum J value
    idx = np.argmax(J)

    # Get the best threshold
    best_threshold = thresholds[idx]


    recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    print(f"Recall: {recall:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score, roc_auc_score, accuracy_score

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df['prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)

    # Normalize the prediction values for each protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    merged_df.drop('prediction', axis=1, inplace=True)

    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()




    def top_residues_within_radius(group, avg_radius):
        group = group.sort_values('pred', ascending=False)
        top_10 = group.iloc[:10]
        top_10_coords = top_10[['Cax', 'Cay', 'Caz']].values
        all_coords = group[['Cax', 'Cay', 'Caz']].values

        max_pred_sum = 0
        best_coords = None

        for coord in top_10_coords:
            distances = np.sqrt(np.sum((all_coords - coord)**2, axis=1))
            current_group = group[distances <= avg_radius]
            current_pred_sum = current_group['pred'].sum()

            if current_pred_sum > max_pred_sum:
                max_pred_sum = current_pred_sum
                best_coords = current_group.index

        group['pred'] = 0

        if best_coords is not None:
            group.loc[best_coords, 'pred'] = 1

        return group

    avg_radius = 15

    merged_df = merged_df.groupby('name').apply(lambda x: top_residues_within_radius(x, avg_radius)).reset_index(drop=True)





    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Filter out rows where 'pred' is NaN
    merged_df_filtered = merged_df.dropna(subset=['pred'])

    # Get true binary labels
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get prediction scores
    pred_scores = merged_df_filtered['pred'].values

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()


    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get false positive rate, true positive rate and thresholds using roc_curve
    fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

    # Calculate Youden's J statistic for each threshold
    J = tpr - fpr

    # Get the index of the maximum J value
    idx = np.argmax(J)

    # Get the best threshold
    best_threshold = thresholds[idx]


    recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    print(f"Recall: {recall:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score, roc_auc_score, accuracy_score

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df['prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)

    # Normalize the prediction values for each protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    merged_df.drop('prediction', axis=1, inplace=True)

    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()




    def top_residues_within_radius(group, avg_radius):
        group = group.sort_values('pred', ascending=False)
        top_10 = group.iloc[:10]
        top_10_coords = top_10[['Cax', 'Cay', 'Caz']].values
        all_coords = group[['Cax', 'Cay', 'Caz']].values

        max_pred_sum = 0
        best_coords = None

        for coord in top_10_coords:
            distances = np.sqrt(np.sum((all_coords - coord)**2, axis=1))
            current_group = group[distances <= avg_radius]
            current_pred_sum = current_group['pred'].sum()

            if current_pred_sum > max_pred_sum:
                max_pred_sum = current_pred_sum
                best_coords = current_group.index

        group['pred'] = 0

        if best_coords is not None:
            group.loc[best_coords, 'pred'] = 1

        return group

    avg_radius = 20

    merged_df = merged_df.groupby('name').apply(lambda x: top_residues_within_radius(x, avg_radius)).reset_index(drop=True)





    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Filter out rows where 'pred' is NaN
    merged_df_filtered = merged_df.dropna(subset=['pred'])

    # Get true binary labels
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get prediction scores
    pred_scores = merged_df_filtered['pred'].values

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()


    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get false positive rate, true positive rate and thresholds using roc_curve
    fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

    # Calculate Youden's J statistic for each threshold
    J = tpr - fpr

    # Get the index of the maximum J value
    idx = np.argmax(J)

    # Get the best threshold
    best_threshold = thresholds[idx]


    recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    print(f"Recall: {recall:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score, roc_auc_score, accuracy_score

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df['prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)

    # Normalize the prediction values for each protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    merged_df.drop('prediction', axis=1, inplace=True)

    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()




    def top_residues_within_radius(group, avg_radius):
        group = group.sort_values('pred', ascending=False)
        top_10 = group.iloc[:10]
        top_10_coords = top_10[['Cax', 'Cay', 'Caz']].values
        all_coords = group[['Cax', 'Cay', 'Caz']].values

        max_pred_sum = 0
        best_coords = None

        for coord in top_10_coords:
            distances = np.sqrt(np.sum((all_coords - coord)**2, axis=1))
            current_group = group[distances <= avg_radius]
            current_pred_sum = current_group['pred'].sum()

            if current_pred_sum > max_pred_sum:
                max_pred_sum = current_pred_sum
                best_coords = current_group.index

        group['pred'] = 0

        if best_coords is not None:
            group.loc[best_coords, 'pred'] = 1

        return group

    avg_radius = 25

    merged_df = merged_df.groupby('name').apply(lambda x: top_residues_within_radius(x, avg_radius)).reset_index(drop=True)





    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

    # Filter out rows where 'pred' is NaN
    merged_df_filtered = merged_df.dropna(subset=['pred'])

    # Get true binary labels
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get prediction scores
    pred_scores = merged_df_filtered['pred'].values

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()


    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get false positive rate, true positive rate and thresholds using roc_curve
    fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

    # Calculate Youden's J statistic for each threshold
    J = tpr - fpr

    # Get the index of the maximum J value
    idx = np.argmax(J)

    # Get the best threshold
    best_threshold = thresholds[idx]


    recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    print(f"Recall: {recall:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

In [ ]:
protein_names = merged_df_filtered['name'].unique()

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        auc = roc_auc_score(true_labels, protein_df['pred'])
        if auc > 0.9:
          print(protein)

In [ ]:
from sklearn.metrics import roc_curve, recall_score

protein_names = merged_df_filtered['name'].unique()

for protein in protein_names:
    protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['pred'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        recall = recall_score(true_labels, (protein_df['pred'] >= best_threshold).astype(int))
        if recall < 0.05:
          print(protein)

In [ ]:
import os
import pandas as pd
from Bio.PDB import PDBParser, PDBIO, is_aa

best_names = [
'2J88_1_L_H_A'
]

best_df = merged_df_filtered[merged_df_filtered['name'].isin(best_names)]
best_df['ResID'] = best_df['ResID'].astype(str)

input_pdb_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v4/fold_{i}/data_test/structures/"
output_pdb_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/"

parser = PDBParser(QUIET=True)
io = PDBIO()

for index, row in best_df.iterrows():
    pdb_filename = row['name'] + '.pdb'
    file_found = False

    for x in range(1, 6):
        input_file = input_pdb_path.format(i=x) + pdb_filename
        if os.path.isfile(input_file):
            file_found = True
            break

    if not file_found:
        print(f"{pdb_filename} not found in any fold directory.")
        continue

    structure = parser.get_structure(row['name'], input_file)

    for model in structure:
        for chain in model:
            for residue in chain:
                if not is_aa(residue, standard=True):
                    continue
                res_id_tuple = residue.get_id()
                res_id = f"{res_id_tuple[0]}{res_id_tuple[1]}{res_id_tuple[2]}".strip()
                matched_rows = best_df[(best_df['name'] == row['name']) & (best_df['ResID'] == res_id)]
                if chain.id in ['L', 'H']:
                    pred_value = 0
                elif not matched_rows.empty:
                    pred_value = matched_rows.iloc[0]['pred']
                else:
                    pred_value = 0
                for atom in residue:
                    atom.set_bfactor(pred_value)

    output_file = os.path.join(output_pdb_path, pdb_filename)
    io.set_structure(structure)
    io.save(output_file)
    print(f"Processed and saved {output_file}")


```pymol
spectrum b, rainbow, minimum=0, maximum=1
color magenta, chain H or chain L
set internal_gui, off
```


In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

accuracy_list = []
recall_list = []
rate_list = np.linspace(0.01, 1.0, 10)

# Create a list to store DataFrames from each fold
all_fold_dfs = []

for rate in rate_list:
    # Process each fold's training set
    for fold in range(1, 6):

        # 1. Load the trained model
        model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
        model_path = os.path.join(model_dir, 'model_cnn.pth')
        model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
        model_cnn.eval()  # Set the model to evaluation mode
        BATCH_SIZE = 500

        # 2. Convert the testing patches into a DataLoader
        testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
        testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
        testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
        testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

        # 3. Loop through the DataLoader and pass each batch to the model
        predictions = []
        with torch.no_grad():
            for batch in testing_loader:
                outputs = model_cnn(batch)
                outputs = outputs.squeeze().cpu().numpy()
                predictions.extend(outputs)

        # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
        binary_predictions = predictions

        # Save predictions to a file
        np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

        # File paths
        input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
        output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

        # Lists to store the data
        protein_names = []
        patch_indices = []

        # Open and read the text file
        with open(input_file_path, 'r') as file:
            lines = file.readlines()
            current_protein_name = None
            for line in lines:
                line = line.strip()
                if line.startswith('Patches for protein'):
                    current_protein_name = line.split(' ')[3][:-1]
                elif line.startswith('indices:'):
                    indices = line.split('[')[1].split(']')[0]
                    protein_names.append(current_protein_name)
                    patch_indices.append(indices)

        # Create a DataFrame
        df_patch = pd.DataFrame({
            'Protein_Name': protein_names,
            'Patch_Indices': patch_indices
        })

        # Save the DataFrame to a CSV file
        df_patch.to_csv(output_file_path, index=False)

        df_patch['Predictions'] = binary_predictions

        df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

        filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

        filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

        # Step 1: Transform df_patch to long format
        df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
        df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

        # Step 2: Merge on Protein_Name and Patch_Indices
        merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

        # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
        merged_df['prediction'] = merged_df['Predictions']
        merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
        merged_df['prediction'].fillna(np.nan, inplace=True)

        # Normalize the prediction values for each protein
        merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
        merged_df.drop('prediction', axis=1, inplace=True)

        df_groupby_cols = merged_df.columns.tolist()[:-1]
        merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()



        def top_clusters_with_highest_pred(group,distance_threshold, rate):
            # Extract coordinates
            coords = group[['Cax', 'Cay', 'Caz']].values

            # Use DBSCAN to cluster residues based on the distance threshold
            clustering = DBSCAN(eps=distance_threshold, min_samples=1).fit(coords)
            group['cluster'] = clustering.labels_

            # Calculate the prediction sum for each cluster
            cluster_pred_sums = group.groupby('cluster')['pred'].max().sort_values(ascending=False)

            # Select the top clusters based on the sum of predictions
            num_clusters = int(len(cluster_pred_sums) * rate)
            top_clusters = cluster_pred_sums.head(num_clusters).index

            # Set 'pred' to 1 for residues in the top clusters, 0 otherwise
            group['pred'] = group['cluster'].apply(lambda x: 1 if x in top_clusters else 0)

            # Drop the 'cluster' column
            group = group.drop(columns=['cluster'])

            return group

        distance_threshold = 4.5

        # Apply the function to each group in the DataFrame
        merged_df = merged_df.groupby('name').apply(lambda x: top_clusters_with_highest_pred(x,distance_threshold,rate)).reset_index(drop=True)


        merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

        # Collect the DataFrame
        all_fold_dfs.append(merged_df)

    # Combine all DataFrames
    combined_df = pd.concat(all_fold_dfs, ignore_index=True)

    merged_df_filtered = combined_df.dropna(subset=['pred'])

    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    # Get false positive rate, true positive rate and thresholds using roc_curve
    fpr, tpr, thresholds = roc_curve(true_labels, merged_df_filtered['pred'])

    # Calculate Youden's J statistic for each threshold
    J = tpr - fpr

    # Get the index of the maximum J value
    idx = np.argmax(J)

    # Get the best threshold
    best_threshold = thresholds[idx]

    # Calculate metrics
    accuracy = accuracy_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    recall = recall_score(true_labels, (merged_df_filtered['pred'] >= best_threshold).astype(int))
    accuracy_list.append(accuracy)
    recall_list.append(recall)

In [ ]:
plt.figure()
plt.plot(rate_list, accuracy_list, label='Accuracy', color='blue')
plt.xlabel('Rate')
plt.ylabel('Score')
plt.title('Accuracy vs Rate')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
plt.plot(rate_list, recall_list, label='Recall', color='orange')
plt.xlabel('Rate')
plt.ylabel('Score')
plt.title('Recall vs Rate')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

rate_list = np.linspace(0.01, 1.0, 10)

# Create a list to store DataFrames from each fold
all_fold_dfs = []

for rate in rate_list:
    # Process each fold's training set
    for fold in range(1, 6):

        # 1. Load the trained model
        model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
        model_path = os.path.join(model_dir, 'model_cnn.pth')
        model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
        model_cnn.eval()  # Set the model to evaluation mode
        BATCH_SIZE = 500

        # 2. Convert the testing patches into a DataLoader
        testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
        testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
        testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
        testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

        # 3. Loop through the DataLoader and pass each batch to the model
        predictions = []
        with torch.no_grad():
            for batch in testing_loader:
                outputs = model_cnn(batch)
                outputs = outputs.squeeze().cpu().numpy()
                predictions.extend(outputs)

        # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
        binary_predictions = predictions

        # Save predictions to a file
        np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

        # File paths
        input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
        output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

        # Lists to store the data
        protein_names = []
        patch_indices = []

        # Open and read the text file
        with open(input_file_path, 'r') as file:
            lines = file.readlines()
            current_protein_name = None
            for line in lines:
                line = line.strip()
                if line.startswith('Patches for protein'):
                    current_protein_name = line.split(' ')[3][:-1]
                elif line.startswith('indices:'):
                    indices = line.split('[')[1].split(']')[0]
                    protein_names.append(current_protein_name)
                    patch_indices.append(indices)

        # Create a DataFrame
        df_patch = pd.DataFrame({
            'Protein_Name': protein_names,
            'Patch_Indices': patch_indices
        })

        # Save the DataFrame to a CSV file
        df_patch.to_csv(output_file_path, index=False)

        df_patch['Predictions'] = binary_predictions

        df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

        filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

        filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

        # Step 1: Transform df_patch to long format
        df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
        df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

        # Step 2: Merge on Protein_Name and Patch_Indices
        merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

        # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
        merged_df['prediction'] = merged_df['Predictions']
        merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
        merged_df['prediction'].fillna(np.nan, inplace=True)

        # Normalize the prediction values for each protein
        merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
        merged_df.drop('prediction', axis=1, inplace=True)

        df_groupby_cols = merged_df.columns.tolist()[:-1]
        merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()



        def top_clusters_with_highest_pred(group,distance_threshold, rate):
            # Extract coordinates
            coords = group[['Cax', 'Cay', 'Caz']].values

            # Use DBSCAN to cluster residues based on the distance threshold
            clustering = DBSCAN(eps=distance_threshold, min_samples=1).fit(coords)
            group['cluster'] = clustering.labels_

            # Calculate the prediction sum for each cluster
            cluster_pred_sums = group.groupby('cluster')['pred'].max().sort_values(ascending=False)

            # Select the top clusters based on the sum of predictions
            num_clusters = int(len(cluster_pred_sums) * rate)
            top_clusters = cluster_pred_sums.head(num_clusters).index

            # Set 'pred' to 1 for residues in the top clusters, 0 otherwise
            group['pred'] = group['cluster'].apply(lambda x: 1 if x in top_clusters else 0)

            # Drop the 'cluster' column
            group = group.drop(columns=['cluster'])

            return group

        distance_threshold = 4.5

        # Apply the function to each group in the DataFrame
        merged_df = merged_df.groupby('name').apply(lambda x: top_clusters_with_highest_pred(x,distance_threshold,rate)).reset_index(drop=True)


        merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

        # Collect the DataFrame
        all_fold_dfs.append(merged_df)

    # Combine all DataFrames
    combined_df = pd.concat(all_fold_dfs, ignore_index=True)

    merged_df_filtered = combined_df.dropna(subset=['pred'])

    # Calculate recall for each protein
    protein_names = merged_df_filtered['name'].unique()
    recall_list = []

    for protein in protein_names:
        protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
        true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
        if len(np.unique(true_labels)) == 2:
            fpr, tpr, thresholds = roc_curve(true_labels, protein_df['pred'])
            J = tpr - fpr
            idx = np.argmax(J)
            best_threshold = thresholds[idx]
            recall = recall_score(true_labels, (protein_df['pred'] >= best_threshold).astype(int))
            recall_list.append(recall)

    # Plot recall distribution
    plt.hist(recall_list, bins=20, edgecolor='k', alpha=0.7)
    plt.xlabel('Recall')
    plt.ylabel('Frequency')
    plt.title('Distribution of recall for each protein using large patch prediction')
    plt.grid(True)
    plt.show()

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import roc_curve, recall_score
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Function to process and plot data with dynamic rate
def plot_recall_distribution(rate):
    # Create a list to store DataFrames from each fold
    all_fold_dfs = []

    # Process each fold's training set
    for fold in range(1, 6):

        # 1. Load the trained model
        model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
        model_path = os.path.join(model_dir, 'model_cnn.pth')
        model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
        model_cnn.eval()  # Set the model to evaluation mode
        BATCH_SIZE = 500

        # 2. Convert the testing patches into a DataLoader
        testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
        testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
        testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
        testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

        # 3. Loop through the DataLoader and pass each batch to the model
        predictions = []
        with torch.no_grad():
            for batch in testing_loader:
                outputs = model_cnn(batch)
                outputs = outputs.squeeze().cpu().numpy()
                predictions.extend(outputs)

        # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
        binary_predictions = predictions

        # Save predictions to a file
        np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

        # File paths
        input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
        output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

        # Lists to store the data
        protein_names = []
        patch_indices = []

        # Open and read the text file
        with open(input_file_path, 'r') as file:
            lines = file.readlines()
            current_protein_name = None
            for line in lines:
                line = line.strip()
                if line.startswith('Patches for protein'):
                    current_protein_name = line.split(' ')[3][:-1]
                elif line.startswith('indices:'):
                    indices = line.split('[')[1].split(']')[0]
                    protein_names.append(current_protein_name)
                    patch_indices.append(indices)

        # Create a DataFrame
        df_patch = pd.DataFrame({
            'Protein_Name': protein_names,
            'Patch_Indices': patch_indices
        })

        # Save the DataFrame to a CSV file
        df_patch.to_csv(output_file_path, index=False)

        df_patch['Predictions'] = binary_predictions

        df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

        filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

        filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

        # Step 1: Transform df_patch to long format
        df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
        df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

        # Step 2: Merge on Protein_Name and Patch_Indices
        merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

        # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
        merged_df['prediction'] = merged_df['Predictions']
        merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
        merged_df['prediction'].fillna(np.nan, inplace=True)

        # Normalize the prediction values for each protein
        merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
        merged_df.drop('prediction', axis=1, inplace=True)

        df_groupby_cols = merged_df.columns.tolist()[:-1]
        merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()

        def top_clusters_with_highest_pred(group, distance_threshold, rate):
            # Extract coordinates
            coords = group[['Cax', 'Cay', 'Caz']].values

            # Use DBSCAN to cluster residues based on the distance threshold
            clustering = DBSCAN(eps=distance_threshold, min_samples=1).fit(coords)
            group['cluster'] = clustering.labels_

            # Calculate the prediction sum for each cluster
            cluster_pred_sums = group.groupby('cluster')['pred'].max().sort_values(ascending=False)

            # Select the top clusters based on the sum of predictions
            num_clusters = int(len(cluster_pred_sums) * rate)
            top_clusters = cluster_pred_sums.head(num_clusters).index

            # Set 'pred' to 1 for residues in the top clusters, 0 otherwise
            group['pred'] = group['cluster'].apply(lambda x: 1 if x in top_clusters else 0)

            # Drop the 'cluster' column
            group = group.drop(columns=['cluster'])

            return group

        distance_threshold = 4.5

        # Apply the function to each group in the DataFrame
        merged_df = merged_df.groupby('name').apply(lambda x: top_clusters_with_highest_pred(x, distance_threshold, rate)).reset_index(drop=True)

        merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

        # Collect the DataFrame
        all_fold_dfs.append(merged_df)

    # Combine all DataFrames
    combined_df = pd.concat(all_fold_dfs, ignore_index=True)

    merged_df_filtered = combined_df.dropna(subset=['pred'])

    # Calculate recall for each protein
    protein_names = merged_df_filtered['name'].unique()
    recall_list = []

    for protein in protein_names:
        protein_df = merged_df_filtered[merged_df_filtered['name'] == protein]
        true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
        if len(np.unique(true_labels)) == 2:
            fpr, tpr, thresholds = roc_curve(true_labels, protein_df['pred'])
            J = tpr - fpr
            idx = np.argmax(J)
            best_threshold = thresholds[idx]
            recall = recall_score(true_labels, (protein_df['pred'] >= best_threshold).astype(int))
            recall_list.append(recall)

    # Plot recall distribution
    plt.figure(figsize=(8, 6))
    plt.hist(recall_list, bins=20, edgecolor='k', alpha=0.7)
    plt.xlabel('Recall')
    plt.ylabel('Frequency')
    plt.title(f'Distribution of Recall for each Protein (rate={rate:.2f})')
    plt.grid(True)
    plt.show()

# Create an interactive slider for rate
interact(plot_recall_distribution, rate=FloatSlider(min=0, max=1.0, step=0.1, value=0.1))

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

auc_list = []
cutoff_list = np.linspace(0, 1, 10)

# Create a list to store DataFrames from each fold
all_fold_dfs = []

for cutoff in cutoff_list:
    # Process each fold's training set
    for fold in range(1, 6):

        # 1. Load the trained model
        model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
        model_path = os.path.join(model_dir, 'model_cnn.pth')
        model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
        model_cnn.eval()  # Set the model to evaluation mode
        BATCH_SIZE = 500

        # 2. Convert the testing patches into a DataLoader
        testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
        testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
        testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
        testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

        # 3. Loop through the DataLoader and pass each batch to the model
        predictions = []
        with torch.no_grad():
            for batch in testing_loader:
                outputs = model_cnn(batch)
                outputs = outputs.squeeze().cpu().numpy()
                predictions.extend(outputs)

        # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
        binary_predictions = predictions

        # Save predictions to a file
        np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

        # File paths
        input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
        output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

        # Lists to store the data
        protein_names = []
        patch_indices = []

        # Open and read the text file
        with open(input_file_path, 'r') as file:
            lines = file.readlines()
            current_protein_name = None
            for line in lines:
                line = line.strip()
                if line.startswith('Patches for protein'):
                    current_protein_name = line.split(' ')[3][:-1]
                elif line.startswith('indices:'):
                    indices = line.split('[')[1].split(']')[0]
                    protein_names.append(current_protein_name)
                    patch_indices.append(indices)

        # Create a DataFrame
        df_patch = pd.DataFrame({
            'Protein_Name': protein_names,
            'Patch_Indices': patch_indices
        })

        # Save the DataFrame to a CSV file
        df_patch.to_csv(output_file_path, index=False)

        df_patch['Predictions'] = binary_predictions

        df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

        filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

        filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

        # Step 1: Transform df_patch to long format
        df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
        df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

        # Step 2: Merge on Protein_Name and Patch_Indices
        merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

        # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
        merged_df['prediction'] = merged_df['Predictions']
        merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
        merged_df['prediction'].fillna(np.nan, inplace=True)

        # Normalize the prediction values for each protein
        merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
        merged_df.drop('prediction', axis=1, inplace=True)

        df_groupby_cols = merged_df.columns.tolist()[:-1]
        merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()

        merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]

        # Collect the DataFrame
        all_fold_dfs.append(merged_df)

    # Combine all DataFrames
    combined_df = pd.concat(all_fold_dfs, ignore_index=True)

    merged_df_filtered = combined_df.dropna(subset=['pred'])

    predicted_labels = np.where(merged_df_filtered['pred'] >= cutoff, 1, 0)

    # Convert 'Status' column to binary values
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)

    auc = roc_auc_score(true_labels, predicted_labels)
    auc_list.append(auc)

In [ ]:
plt.figure()
plt.plot(cutoff_list, auc_list, label='AUC', color='blue')
plt.xlabel('Cutoff')
plt.ylabel('Score')
plt.title('AUC vs Cutoff')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)


# Create a list to store DataFrames from each fold
all_fold_dfs = []


# Process each fold's training set
for fold in range(1, 6):

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Convert the testing patches into a DataLoader
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)
    testing_loader = DataLoader(testing_patches_tensor, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Loop through the DataLoader and pass each batch to the model
    predictions = []
    with torch.no_grad():
        for batch in testing_loader:
            outputs = model_cnn(batch)
            outputs = outputs.squeeze().cpu().numpy()
            predictions.extend(outputs)

    # 4. Analyze predictions (e.g., convert to binary class labels, save to file, etc.)
    binary_predictions = predictions

    # Save predictions to a file
    np.save(os.path.join(testing_dir, 'predictions.npy'), np.array(binary_predictions))

    # File paths
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Lists to store the data
    protein_names = []
    patch_indices = []

    # Open and read the text file
    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    # Create a DataFrame
    df_patch = pd.DataFrame({
        'Protein_Name': protein_names,
        'Patch_Indices': patch_indices
    })

    # Save the DataFrame to a CSV file
    df_patch.to_csv(output_file_path, index=False)

    df_patch['Predictions'] = binary_predictions

    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')

    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))

    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Step 1: Transform df_patch to long format
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Step 2: Merge on Protein_Name and Patch_Indices
    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Step 3: Add the 'prediction' column and fill NaN values for residues that don't have a prediction
    merged_df['prediction'] = merged_df['Predictions']
    merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'], inplace=True)
    merged_df['prediction'].fillna(np.nan, inplace=True)

    # Normalize the prediction values for each protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
    merged_df.drop('prediction', axis=1, inplace=True)

    df_groupby_cols = merged_df.columns.tolist()[:-1]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()



    merged_df = merged_df[merged_df['SaSaRate'] >= 0.1]


    # Collect the DataFrame
    all_fold_dfs.append(merged_df)

# Combine all DataFrames
combined_df = pd.concat(all_fold_dfs, ignore_index=True)

merged_df_filtered = combined_df.dropna(subset=['pred'])

true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)
precision, recall, thresholds = precision_recall_curve(true_labels, merged_df_filtered['pred'])

pr_auc = auc(recall, precision)

# Plotting the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'Precision-Recall curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()


In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):
    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Load testing data and metadata
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')

    # Load patches and convert to tensor
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)

    # Load patch metadata (protein names and residue indices)
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    output_file_path = os.path.join(testing_dir, 'testing_patch_index.csv')

    # Parse or load existing metadata
    if not os.path.exists(output_file_path):
        # Parsing code from original implementation
        protein_names = []
        patch_indices = []
        with open(input_file_path, 'r') as file:
            lines = file.readlines()
            current_protein_name = None
            for line in lines:
                line = line.strip()
                if line.startswith('Patches for protein'):
                    current_protein_name = line.split(' ')[3][:-1]
                elif line.startswith('indices:'):
                    indices = line.split('[')[1].split(']')[0]
                    protein_names.append(current_protein_name)
                    patch_indices.append(indices)
        df_patch = pd.DataFrame({'Protein_Name': protein_names, 'Patch_Indices': patch_indices})
        df_patch.to_csv(output_file_path, index=False)
    else:
        df_patch = pd.read_csv(output_file_path)

    # 3. Group patches by protein
    grouped = df_patch.groupby('Protein_Name')

    # 4. Process each protein individually
    all_predictions = []
    for protein_name, group in grouped:
        # Get indices for this protein's patches
        indices = group.index.tolist()
        protein_patches = testing_patches_tensor[indices]

        # Create DataLoader for this protein
        protein_loader = DataLoader(protein_patches, batch_size=BATCH_SIZE, shuffle=False)

        # Make predictions
        protein_predictions = []
        with torch.no_grad():
            for batch in protein_loader:
                outputs = model_cnn(batch)
                outputs = outputs.squeeze().cpu().numpy()
                protein_predictions.extend(outputs)

        # Store predictions with metadata
        protein_df = group.copy()
        protein_df['Predictions'] = protein_predictions
        all_predictions.append(protein_df)

    # Combine all protein predictions
    df_patch = pd.concat(all_predictions)

    # 5. Merge predictions with original data
    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')
    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))
    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    # Convert patch indices to integer list
    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    # Merge predictions
    merged_df = pd.merge(filtered_combined_df,
                        df_patch_long,
                        left_on=['name', 'ResID'],
                        right_on=['Protein_Name', 'Patch_Indices'],
                        how='left')

    # Post-processing
    merged_df['prediction'] = merged_df['Predictions']
    merged_df = merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'])

    # Normalize predictions per protein
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(
        lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)

    # Aggregate results
    df_groupby_cols = merged_df.columns.tolist()[:-2]  # Exclude prediction and pred columns
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()

    # Filter and save
    merged_df_filtered = merged_df.dropna(subset=['pred'])
    merged_df_filtered.to_csv(os.path.join(save_dir, f'data_{fold}_combined_pred.csv'), index=False)

    # 6. Evaluation metrics
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)
    pred_scores = merged_df_filtered['pred'].values

    # ROC Curve
    fpr, tpr, thresholds = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Fold {fold} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Fold {fold} ROC Curve')
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(save_dir, f'fold_{fold}_roc.png'))
    plt.close()

    # Calculate metrics
    J = tpr - fpr
    idx = np.argmax(J)
    best_threshold = thresholds[idx]

    y_pred = (pred_scores >= best_threshold).astype(int)

    metrics = {
        'Accuracy': accuracy_score(true_labels, y_pred),
        'Precision': precision_score(true_labels, y_pred),
        'Recall': recall_score(true_labels, y_pred),
        'F1': f1_score(true_labels, y_pred),
        'AUC': roc_auc,
        'Best Threshold': best_threshold
    }

    print(f'\nFold {fold} Metrics:')
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}' if k != 'Best Threshold' else f'{k}: {v:.3f}')

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, auc, recall_score, accuracy_score, precision_score,
                             f1_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve)

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/combined_results/'

os.makedirs(save_dir, exist_ok=True)

# Process each fold's training set
for fold in range(1, 6):
    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'model_cnn.pth')
    model_cnn = torch.load(model_path, map_location=torch.device('cpu'))
    model_cnn.eval()  # Set the model to evaluation mode
    BATCH_SIZE = 500

    # 2. Load testing data and metadata
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    testing_patches = np.load(os.path.join(testing_dir, 'cnn_testing_patch_3points.npy'))
    testing_patches_tensor = torch.Tensor(testing_patches).reshape(len(testing_patches), 1, 15, 20)

    # Load patch metadata
    df_patch = pd.read_csv(os.path.join(testing_dir, 'testing_patch_index.csv'))
    grouped = df_patch.groupby('Protein_Name')

    # 3. Process each protein individually
    all_predictions = []
    for protein_name, group in grouped:
        indices = group.index.tolist()
        protein_patches = testing_patches_tensor[indices]
        protein_loader = DataLoader(protein_patches, batch_size=BATCH_SIZE, shuffle=False)

        # Make predictions
        protein_predictions = []
        with torch.no_grad():
            for batch in protein_loader:
                outputs = model_cnn(batch)
                outputs = outputs.squeeze().cpu().numpy()
                protein_predictions.extend(outputs)

        protein_df = group.copy()
        protein_df['Predictions'] = protein_predictions
        all_predictions.append(protein_df)

    df_patch = pd.concat(all_predictions)

    # 4. Merge predictions with original data
    df_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test/')
    filtered_combined_df = pd.read_csv(os.path.join(df_dir, 'all_data_with_SI.csv'))
    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    df_patch_long = df_patch.assign(Patch_Indices=df_patch['Patch_Indices'].str.split(', ')).explode('Patch_Indices')
    df_patch_long['Patch_Indices'] = df_patch_long['Patch_Indices'].astype(int)

    merged_df = pd.merge(filtered_combined_df, df_patch_long, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')
    merged_df['prediction'] = merged_df['Predictions']
    merged_df = merged_df.drop(columns=['Protein_Name', 'Patch_Indices', 'Predictions'])
    merged_df['pred'] = merged_df.groupby('name')['prediction'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)

    df_groupby_cols = merged_df.columns.tolist()[:-2]
    merged_df = merged_df.groupby(df_groupby_cols, as_index=False)['pred'].max()

    merged_df_filtered = merged_df.dropna(subset=['pred'])
    merged_df_filtered.to_csv(os.path.join(save_dir, f'data_{fold}_combined_pred.csv'), index=False)

    # 5. Evaluation metrics
    true_labels = np.where(merged_df_filtered['Status'] == 'Interacting', 1, 0)
    pred_scores = merged_df_filtered['pred'].values

    # ROC Curve
    fpr, tpr, thresholds = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Fold {fold} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Fold {fold} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(os.path.join(save_dir, f'fold_{fold}_roc.png'))
    plt.close()

    # Best threshold using Youden's J statistic
    J = tpr - fpr
    idx = np.argmax(J)
    best_threshold = thresholds[idx]
    y_pred = (pred_scores >= best_threshold).astype(int)

    # Additional Metrics
    bac = balanced_accuracy_score(true_labels, y_pred)
    mcc = matthews_corrcoef(true_labels, y_pred)
    precision, recall, _ = precision_recall_curve(true_labels, pred_scores)
    pr_auc = auc(recall, precision)
    nantigens = merged_df_filtered['name'].nunique()
    f_predicted = np.mean(y_pred) * 100

    # Print Metrics
    metrics = {
        'Accuracy': accuracy_score(true_labels, y_pred),
        'Precision': precision_score(true_labels, y_pred),
        'Recall': recall_score(true_labels, y_pred),
        'F1': f1_score(true_labels, y_pred),
        'AUC': roc_auc,
        'PR-AUC': pr_auc,
        'BAC': bac,
        'MCC': mcc,
        'Best Threshold': best_threshold,
        'Nantigens': nantigens,
        'Fpredicted': f_predicted
    }

    print(f'\nFold {fold} Metrics:')
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}' if k not in ['Best Threshold', 'Nantigens', 'Fpredicted'] else f'{k}: {v:.3f}%')
